Now, we can look into the synthesis of a conditional order, a.k.a. a pruning rule that is dependent to the takeoff times $t_i$ and $t_j$ directly.

In [1]:
from synthesis.grammar import *
from synthesis.synth import *
from synthesis import *

problem = make_rsp_swap_problem(timeout_ms = 10000, objective_name = 'makespan')

log(f"Symbol Set: [{', '.join(symbol.name.replace('_', '') for symbol in problem.symbols)}]")

# Construct the grammar.
conj = NonTerminal("Rule", sort = problem.env.bool_sort)
cmp = NonTerminal("Atom", sort = problem.env.bool_sort)

by_name = {symbol.name: symbol for symbol in problem.symbols}

flat = lambda xs: [y for x in xs for y in (flat(x) if isinstance(x, list) else [x])]
names = set(by_name)

prefixes = sorted({name[:-2] for name in names if name.endswith("_i") and not name.startswith("D_")})
comparable_pairs = [
    *[(f"{p}_i", f"{p}_j") for p in prefixes if f"{p}_j" in names],
    *[pair for pair in (("D_i_x", "D_j_x"), ("D_x_i", "D_x_j")) if pair[0] in names and pair[1] in names],
]

grammar = Grammar(
    nonterminals=(conj, cmp),
    terminals=problem.symbols,
    start=conj,
    productions=(
        conj >> (cmp | (cmp & cmp)),
        cmp >> Choice(tuple(flat(
            [[by_name[l] <= by_name[r], by_name[r] <= by_name[l], by_name[l].eq(by_name[r])] for l, r in comparable_pairs]
        ))),
    ),
)
log(f"Visualising Context-Free Grammar for SyGuS:\n{grammar.vis()}\n")

# Binds Terminal symbols to problem variables.
set_smt_env(
    symbol_table={symbol.name: symbol.formal for symbol in problem.symbols},
)
grammar = grammar.to_cvc5(problem.env.solver)


# Run Synthesis
result = synthesize_pruning_rule(
    problem,
    grammar=grammar,
    require_nonvacuous=True,
)



[16:13:47] Symbol Set: [Ri, Rj, LTi, LTj, LCi, LCj, ETi, ETj, ECi, ECj, Bi, Bj, Ci, Cj, Dix, Djx, Dxi, Dxj, Ti, Tj, T'i, T'j]
[16:13:47] Visualising Context-Free Grammar for SyGuS:
<Rule> ::= <Atom> | <Atom> & <Atom>
<Atom> ::= B_i <= B_j | B_j <= B_i | B_i = B_j | C_i <= C_j | C_j <= C_i | C_i = C_j | EC_i <= EC_j | EC_j <= EC_i |
           EC_i = EC_j | ET_i <= ET_j | ET_j <= ET_i | ET_i = ET_j | LC_i <= LC_j | LC_j <= LC_i | LC_i = LC_j | LT_i <=
           LT_j | LT_j <= LT_i | LT_i = LT_j | R_i <= R_j | R_j <= R_i | R_i = R_j | T_i <= T_j | T_j <= T_i | T_i = T_j
           | T'_i <= T'_j | T'_j <= T'_i | T'_i = T'_j | D_i_x <= D_j_x | D_j_x <= D_i_x | D_i_x = D_j_x | D_x_i <=
           D_x_j | D_x_j <= D_x_i | D_x_i = D_x_j

[16:13:47] Invoking Synthesis (This may take some time)
[16:13:47] Invoking Synthesis (This may take some time)
[16:13:52] Synthesis Complete, Result: (SOLUTION):

(
  (define-fun prune ((R_i Int) (R_j Int) (LT_i Int) (LT_j Int) (LC_i Int) (LC_j Int) (ET_i 